##### Nesta etapa, o objetivo ? executar todas as a??es necess?rias para preparar o DataFrame original para passar pelo processo de modelagem.
        Dessa forma, algumas etapas s?o necess?rias:
                Limpeza e tratamento de dados faltantes e outliers, se necess?rio;
                Transforma??o de dados, como a normaliza??o e padroniza??o;
                Feature engineering, com a cria??o de rela??es com dados e a verifica??o de inconsist?ncias;
                Resultado: Um novo DF preparado para a etapa de modelagem. 

### Importando os dados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# proximos passos: 
# pre processamento - tratamento de nulos, desenvolvimento das ideias das novas variaveis (features) (cri?-las postoeriormente permite cross-validation...),
# pipeline do treinamento do modelo -> tratamento de outliers, encoder, normaliza??o, cria??o propriamente dita da feature
# entrada da base original -> pre processamento -> base de saida preparada para modelagem

In [ ]:
train_df = pd.read_csv(r"C:\Users\Fernando Costa\Desktop\Arquivos Lucas\DS\Projetos GITHUB\Titanic\Base_dados\train.csv")
df_model = train_df.copy()
df_model.head(5)

In [ ]:
# Executando uma breve verifica??o se as outras colunas (al?m de PassengerID, Name e Ticket) s?o ou n?o relevantes para a predi??o de dados:
num_rows = df_model.shape[0]
# N?mero total de linhas no DataFrame
print(f"O DF tem {num_rows} linhas.")
# Calculando o n?mero de valores ?nicos em cada coluna
unique_counts = df_model.nunique()
print(unique_counts)

? Agora, removeremos as colunas dispens?veis para o nosso modelo, j? que n?o carregam caracteristica preditora, pois tem muitas informa??es "?nicas", al?m da pr?pria an?lise do que cada uma delas representa, como por exemplo 'PassengerId', 'Name' ou 'Cabin.

In [ ]:
# Executando uma limpeza b?sica dos dados, excluindo as colunas que n?o auxiliam na predi??o:
df_model.drop(columns=["PassengerId", "Name", "Ticket"], inplace= True)

## Data Cleaning

### Tratando os valores nulos

Como visto na etapa do EDA, existiam valores nulos nas colunas de 'Age', 'Cabin' e 'Embarked', assim, preencheremos seus nulos respectivamente com valores imposs?veis de serem alcan?ados por esses par?metros, respeitando o dtype das colunas.

In [ ]:
df_model["Age"] = df_model["Age"].fillna(-1)

In [ ]:
# Preenchendo os valores nulos da coluna "Cabin" com dados sint?ticos (NA)
df_model["Cabin"] = df_model["Cabin"].fillna('NA')

In [ ]:
df_model["Embarked"] = df_model["Embarked"].fillna("NA")
# Verificando as modifica??es realizadas no data frame original:

In [ ]:
# Neste caso h? apenas dados nulos em colunas espec?ficas, por?m, em nova amostras podem haver dados nulos em outras colunsa, assim, ? necess?rio garantir que o projeto saiba lidar com isso.
# Desta forma, a substitui??o dos valores nulos ser? realizada com base no dtype da coluna:

# Criando um dicion?rio para preencher os valores de acordo com o tipo da coluna:
fill_values = {col: -1 if df_model[col].dtype in ['int64', 'float64'] else 'NA' for col in df_model.columns}

# Usando fillna() com o dicion?rio
df_model.fillna(value=fill_values, inplace=True)


## Feature Engineering

Introdu??o de tr?s novas m?tricas ao df original que podem auxiliar a previs?o do modelo, ap?s segrega??o de train_test_split, para evitar data leaked.

In [ ]:
# Devem ser utilizados os dados da base de treino, para os c?lculos, independente de qual base estiver criando a coluna.
# 'Simulando' a cria??o do train_test_split:
from sklearn.model_selection import train_test_split
df_train, df_test = train_test_split(df_model, random_state= 15)

A primeira m?trica ser? a Taxa paga pelo passageiro em rela??o a taxa m?dia paga pela sua classe. 

In [ ]:
# Encontrando a Taxa m?dia de cada classe:
taxa_media_classe = df_train.groupby("Pclass")["Fare"].mean()  

# Agora, uma nova coluna deve ser criada: "Taxa_PelaMedia", em ambos os DataSets
df_train["Taxa_PelaMedia"] = df_train["Fare"] / df_train["Pclass"].map(taxa_media_classe)
df_test["Taxa_PelaMedia"] = df_test["Fare"] / df_train["Pclass"].map(taxa_media_classe)

M?trica da Taxa multiplicada a quantidade de irm?os ou conjuges.

In [ ]:
df_train["Taxa_x_SibSp"] = df_train["Fare"] * df_train["SibSp"]
df_test["Taxa_x_SibSp"] = df_test["Fare"] * df_test["SibSp"] # Aqui pode-se utilizar os pr?prio dados pois ? um c?lculo direto

M?trica que transforma "Sex" em uma feature num?rica bin?ria.

In [ ]:
df_train['is_woman'] = (df_train['Sex'] == 'female').astype(int) # Quando a linha atual tiver "female" na coluna "Sex", atribuir 1, se n?o, atribui 0
df_test['is_woman'] = (df_test['Sex'] == 'female').astype(int)

As tr?s features anteriores poderiam ser executadas na etapa de modelagem, por?m, para fins didaticos, pela limita??o desta base de dados, utilizaremos apenas as duas ultimas, n?o sendo necess?rio guardar nenhuma outra informa??o.

Assim, aplicaremos a segunda e terceira feature ao df_model:

In [ ]:
df_model["Taxa_x_SibSp"] = df_model["Fare"] * df_model["SibSp"]
df_model['is_woman'] = (df_model['Sex'] == 'female').astype(int)
df_model = df_model.drop(columns=['Sex'])

## Pr? Enconding

Como foi visto no EDA, a coluna "Pclass" est? com o dtype (int64) errado, como ? uma vari?vel categ?rica, deve ser corrigida:

In [ ]:
# Corrigindo o dtype da coluna 'Pclass' para object:
df_model['Pclass'] = df_model['Pclass'].astype(str)

# Verificando a corre??o realizada
df_model.dtypes 

## Salvando os dados

In [ ]:
# Utilizando a extens?o parquet para salvar os datasets: 
df_model.to_parquet('df_model.parquet')

### Conclus?o

Ap?s toda a etapa de pr? processamento ser realizada, incluindo a cria??o do c?digo com as etapas resumidas para aplica??o em novos dados, a pr?xima etapa ? a modelagem, onde a execu??o dos modelos ocorrer?.